# Практикум 11. Сходство текстов: косинусное сходство, индекс Жаккара и расстояние Левенштейна

В лаборатории 11 вы заполняли таблицу Левенштейна клетка за клеткой, рисовали диаграмму Венна для Жаккара и угол между векторами TF-IDF. Здесь сделаем то же самое в коде:

1. реализуем расстояние Левенштейна динамическим программированием, напечатаем таблицу и восстановим цепочку правок; сверимся с `nltk.edit_distance`;
2. посчитаем индекс Жаккара по словам, по основам и по символьным n-граммам и увидим, как словоизменение «ломает» сравнение по словам;
3. соберём TF-IDF вручную, разложим косинус на вклады отдельных слов и сверимся со `scikit-learn`;
4. сравним три меры на шести типичных парах предложений и применим их к двум задачам: исправлению опечаток и поиску почти-дубликатов.

*Практикум к лаборатории 11 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Расстояние Левенштейна

Расстояние Левенштейна — минимальное число правок (вставка, удаление, замена символа), превращающих строку $a$ в строку $b$. Его считают динамическим программированием: $D[i][j]$ — расстояние между первыми $i$ символами $a$ и первыми $j$ символами $b$:

$$D[i][j] = \min\big(D[i-1][j] + c_{del},\; D[i][j-1] + c_{ins},\; D[i-1][j-1] + c_{sub}\cdot[a_i \ne b_j]\big)$$

Первая строка и первый столбец — расстояние до пустой строки ε. Ответ стоит в правом нижнем углу. Чтобы узнать сами правки, идём из угла назад (backtrace), каждый раз выбирая клетку, из которой пришёл минимум. Пример тот же, что в лаборатории: «кошка» → «окошко».

In [ ]:
import numpy as np

def levenshtein_table(a, b, sub_cost=1, ins_cost=1, del_cost=1):
    # D[i, j] = расстояние между a[:i] и b[:j]
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1))
    D[:, 0] = np.arange(n + 1) * del_cost   # a[:i] -> пусто: только удаления
    D[0, :] = np.arange(m + 1) * ins_cost   # пусто -> b[:j]: только вставки
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = D[i - 1, j - 1] + (0 if a[i - 1] == b[j - 1] else sub_cost)
            D[i, j] = min(D[i - 1, j] + del_cost,   # удалить a[i-1]
                          D[i, j - 1] + ins_cost,   # вставить b[j-1]
                          diag)                     # совпадение или замена
    return D

def print_table(a, b, D):
    print("     " + " ".join(f"{c:>3}" for c in "ε" + b))
    for i, row in enumerate(D):
        label = "ε" if i == 0 else a[i - 1]
        print(f"{label:>3}  " + " ".join(f"{v:>3g}" for v in row))

def backtrace(a, b, D, sub_cost=1, ins_cost=1, del_cost=1):
    # идём из правого нижнего угла назад и записываем операции
    i, j = len(a), len(b)
    ops, path = [], [(i, j)]
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (0 if a[i - 1] == b[j - 1] else sub_cost):
            ops.append(("=" if a[i - 1] == b[j - 1] else "sub", a[i - 1], b[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + del_cost:
            ops.append(("del", a[i - 1], "-"))
            i -= 1
        else:
            ops.append(("ins", "-", b[j - 1]))
            j -= 1
        path.append((i, j))
    return ops[::-1], path[::-1]

OP_NAMES = {"sub": "замена", "del": "удаление", "ins": "вставка"}

def show_alignment(a, b, **costs):
    D = levenshtein_table(a, b, **costs)
    ops, _ = backtrace(a, b, D, **costs)
    print(" ".join(x for _, x, _ in ops))
    print(" ".join({"=": "|", "sub": "*"}.get(op, " ") for op, _, _ in ops))
    print(" ".join(y for _, _, y in ops))
    edits = [f"{OP_NAMES[op]} {x}→{y}" if op == "sub" else f"{OP_NAMES[op]} «{x if op == 'del' else y}»"
             for op, x, y in ops if op != "="]
    print(f"d({a}, {b}) = {D[-1, -1]:g}:", "; ".join(edits))

a, b = "кошка", "окошко"
D = levenshtein_table(a, b)
print_table(a, b, D)
print()
show_alignment(a, b)

Ещё пример и проверка библиотекой. `nltk.edit_distance` считает то же самое; параметр `transpositions=True` добавляет четвёртую операцию — перестановку соседних букв (расстояние Дамерау–Левенштейна), а `substitution_cost=2` делает замену дороже (тогда замена = удаление + вставка). Обратите внимание: «ё» и «е» для компьютера — разные символы.


В конце ячейки — таблица как картинка: цвет — значение $D[i][j]$, оранжевая линия — путь backtrace. Диагональный шаг — совпадение или замена, горизонтальный — вставка, вертикальный — удаление.

In [ ]:
import nltk

show_alignment("ёлка", "елка")
print()

pairs_lev = [("кошка", "окошко"), ("спасибо", "спсаибо"), ("сходство", "сходсво"),
             ("лингвистика", "лингвист"), ("ток", "кот"), ("ёлка", "елка")]
print(f"{'':<26}{'наше':>8}{'nltk':>6}{'+перест.':>8}{'sub=2':>7}")
for a, b in pairs_lev:
    mine = levenshtein_table(a, b)[-1, -1]
    ref = nltk.edit_distance(a, b)
    dl = nltk.edit_distance(a, b, transpositions=True)
    sub2 = levenshtein_table(a, b, sub_cost=2)[-1, -1]
    assert mine == ref and sub2 == nltk.edit_distance(a, b, substitution_cost=2)
    print(f"{a + ' / ' + b:<26}{mine:>8g}{ref:>6}{dl:>8}{sub2:>7g}")

import matplotlib.pyplot as plt

a, b = "кошка", "окошко"
D = levenshtein_table(a, b)
_, path = backtrace(a, b, D)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(D, cmap="Blues")
for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        ax.text(j, i, f"{D[i, j]:g}", ha="center", va="center",
                color="white" if D[i, j] > 0.6 * D.max() else "black")
pi, pj = zip(*path)
ax.plot(pj, pi, color="#D9480F", lw=2.5, marker="o")
ax.set_xticks(range(len(b) + 1))
ax.set_xticklabels(["ε"] + list(b))
ax.set_yticks(range(len(a) + 1))
ax.set_yticklabels(["ε"] + list(a))
ax.xaxis.tick_top()
ax.set_title("Таблица Левенштейна и путь backtrace")
plt.show()

## 2. Индекс Жаккара: слова, основы, символьные n-граммы

Индекс Жаккара сравнивает два **множества**: $J(A,B) = \dfrac{|A \cap B|}{|A \cup B|}$. Он равен 1 для одинаковых множеств и 0 для непересекающихся. Что считать элементами множества — решаем мы: слова, основы слов или символьные n-граммы (кусочки по $n$ букв; `#` отмечает границу слова).

Возьмём два предложения с почти одинаковым смыслом, но разными формами слов.


Затем проверим, помогает ли нормализация: сведём слова к основам стеммером Snowball из `nltk` (он работает по правилам и не требует скачивания данных).

In [ ]:
import re
from nltk.metrics.distance import jaccard_distance

def tokens(text):
    return re.findall(r"[а-яёa-zʻʼ'-]+", text.lower())

def jaccard(A, B):
    A, B = set(A), set(B)
    return len(A & B) / len(A | B) if (A | B) else 1.0

def char_ngrams(text, n=3):
    # символьные n-граммы каждого слова с границами #
    grams = set()
    for w in tokens(text):
        w = "#" + w + "#"
        grams |= {w[i:i + n] for i in range(len(w) - n + 1)}
    return grams

s1 = "Студенты читают новые книги в библиотеке"
s2 = "Студент читает новую книгу в библиотеке"
A, B = set(tokens(s1)), set(tokens(s2))
print("A ∩ B =", sorted(A & B))
print("A ∪ B =", sorted(A | B))
print(f"Жаккар по словам: {len(A & B)}/{len(A | B)} = {jaccard(A, B):.3f}")
print(f"nltk:  1 - jaccard_distance = {1 - jaccard_distance(A, B):.3f}")
print()
for n in (2, 3, 4):
    GA, GB = char_ngrams(s1, n), char_ngrams(s2, n)
    print(f"Жаккар по n-граммам n={n}: {len(GA & GB):>2}/{len(GA | GB):<3} = {jaccard(GA, GB):.3f}")
print("общие триграммы (первые 12):", sorted(char_ngrams(s1, 3) & char_ngrams(s2, 3))[:12])

from nltk.stem.snowball import SnowballStemmer
stem = SnowballStemmer("russian").stem      # отрезает окончания и суффиксы по правилам

for w1, w2 in zip(tokens(s1), tokens(s2)):
    print(f"{w1:<12} → {stem(w1):<11} {w2:<12} → {stem(w2)}")
SA, SB = {stem(w) for w in tokens(s1)}, {stem(w) for w in tokens(s2)}
print(f"\nЖаккар по основам: {len(SA & SB)}/{len(SA | SB)} = {jaccard(SA, SB):.3f}")

Как результат зависит от $n$? Для сравнения добавим узбекскую пару: в агглютинативном языке одно слово несёт число, принадлежность и падеж («kitob-lar-ni»), поэтому любое изменение грамматики меняет всё слово как строку. По словам совпадает мало, а символьные n-граммы сохраняют общие основы. При $n=1$ почти все буквы общие (сходство завышено), при больших $n$ n-граммы становятся «почти словами».

In [ ]:
import matplotlib.pyplot as plt

other = ("Talabalar kutubxonada yangi kitoblarni oʻqidi", "Talaba kutubxonadagi yangi kitobni oʻqidi")   # узбекская пара с тем же смыслом
examples = {"русская пара": (s1, s2), "узбекская пара": other}
ns = range(1, 7)
fig, ax = plt.subplots(figsize=(7, 3.5))
for (label, (x, y)), color in zip(examples.items(), ["#2B55C9", "#17907E"]):
    ax.plot(ns, [jaccard(char_ngrams(x, n), char_ngrams(y, n)) for n in ns], marker="o", color=color,
            label=label + ": n-граммы")
    ax.axhline(jaccard(tokens(x), tokens(y)), ls="--", color=color, label=label + ": слова")
ax.set_xlabel("n")
ax.set_ylabel("индекс Жаккара")
ax.set_ylim(0, 1)
ax.set_title("Жаккар по символьным n-граммам и по словам")
ax.legend(fontsize=8)
plt.show()

## 3. Косинусное сходство и TF-IDF

Текст превращаем в вектор: по оси на каждое слово словаря, значение — вес TF-IDF: $\text{tf}(t,d)\cdot \text{idf}(t)$, где $\text{idf}(t) = \ln\frac{1+N}{1+\text{df}(t)} + 1$ (так считает `scikit-learn`; $N$ — число документов, df — в скольких документах встречается слово). Редкие слова получают больший вес. Косинус угла между векторами:

$$\cos(\vec a, \vec b) = \frac{\sum_i a_i b_i}{\lVert \vec a\rVert\, \lVert \vec b\rVert}$$

Ниже шесть пар предложений, на которых меры ведут себя по-разному (как в таблице лаборатории). Посчитаем TF-IDF вручную и разложим косинус на вклады слов для пары «отрицание».


В конце ячейки сверимся со `scikit-learn`: `TfidfVectorizer` (с нашим токенизатором) и `cosine_similarity`. Числа должны совпасть.

In [ ]:
pairs = [
    ("порядок слов",   "Студент быстро решил трудную задачу", "Трудную задачу быстро решил студент"),
    ("опечатки",       "Завтра в Самарканде будет тёплая погода", "Завтра в Самарканде будит тёплая пагода"),
    ("словоизменение", "Студенты читают новые книги в библиотеке", "Студент читает новую книгу в библиотеке"),
    ("синонимы",       "Врач посоветовал больше гулять", "Доктор рекомендовал чаще бывать на свежем воздухе"),
    ("отрицание",      "Фильм мне очень понравился", "Фильм мне совсем не понравился"),
    ("другая тема",    "Цены на хлопок выросли на бирже", "Футбольный матч перенесли на субботу"),
]

from collections import Counter

docs = [s for _, a, b in pairs for s in (a, b)]
tok_docs = [tokens(d) for d in docs]
vocab = sorted({w for d in tok_docs for w in d})
N = len(docs)
df = Counter(w for d in tok_docs for w in set(d))
# сглаженный idf, как в scikit-learn
idf = {w: np.log((1 + N) / (1 + df[w])) + 1 for w in vocab}

def tfidf_vector(toks):
    tf = Counter(toks)
    v = np.array([tf[w] * idf[w] for w in vocab])
    return v / np.linalg.norm(v)          # L2-нормировка: длина вектора = 1

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

k = 4                                     # пара «отрицание»
name, a, b = pairs[k]
va, vb = tfidf_vector(tokens(a)), tfidf_vector(tokens(b))
print(a, "|", b)
print(f"   {'слово':<12} {'idf':>5}   a_i × b_i")
for i, w in enumerate(vocab):
    if va[i] * vb[i] > 0:
        print(f"   {w:<12} {idf[w]:5.2f}   {va[i]:.3f} × {vb[i]:.3f} = {va[i] * vb[i]:.3f}")
print(f"cos = {cosine(va, vb):.3f}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(tokenizer=tokens, token_pattern=None, lowercase=False)
X = vec.fit_transform(docs)
assert list(vec.get_feature_names_out()) == vocab
assert np.allclose(X.toarray()[2 * k], va)        # векторы совпадают с нашими
S = cosine_similarity(X)
print(f"sklearn: {S[2 * k, 2 * k + 1]:.3f}   вручную: {cosine(va, vb):.3f}")

Теперь сравним все меры на шести парах. Lev — нормированное сходство по Левенштейну $1 - d/\max(|a|,|b|)$ на целых строках; J — Жаккар; cos — косинус TF-IDF по словам и по символьным n-граммам ($n$ = 2…4).

In [ ]:
import pandas as pd

char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))   # n-граммы символов внутри слов, n = 2..4
Sc = cosine_similarity(char_vec.fit_transform(docs))

rows = []
for k, (name, a, b) in enumerate(pairs):
    d = nltk.edit_distance(a.lower(), b.lower())
    rows.append({
        "пара": name,
        "Lev": 1 - d / max(len(a), len(b)),
        "J слова": jaccard(tokens(a), tokens(b)),
        "J 3-граммы": jaccard(char_ngrams(a, 3), char_ngrams(b, 3)),
        "cos слова": S[2 * k, 2 * k + 1],
        "cos символы": Sc[2 * k, 2 * k + 1],
    })
table = pd.DataFrame(rows).set_index("пара").round(2)
print(table.to_string())

Что видно из таблицы:

- **порядок слов**: мешок слов не видит порядка (J и cos по словам = 1), а Левенштейн по строке заметно падает;
- **опечатки** и **словоизменение**: меры по словам проседают, символьные держатся высоко;
- **синонимы**: меры по словам равны нулю, остальные на уровне пары «другая тема», хотя смысл тот же;
- **отрицание**: сходство высокое, хотя смысл противоположный;
- **другая тема**: всё низкое, но не ноль — общие служебные слова («на») и буквы дают «шум»; Левенштейн даже для несвязанных строк держится около 0.2–0.3.

### Как это выглядит на настоящей модели

Нейросетевые эмбеддинги предложений сравнивают смысл, а не буквы. Модель `intfloat/multilingual-e5-small` (≈470 МБ, поддерживает русский и узбекский) переводит предложение в вектор; дальше тот же косинус. Авторы модели советуют добавлять к тексту префикс `"query: "` и предупреждают, что косинусы у неё обычно лежат в диапазоне 0.7–1.0, поэтому важно сравнивать пары между собой, а не с порогом 0.5. Проверьте, поднимется ли пара «синонимы» и распознает ли модель «отрицание». Ячейки работают только в Colab (нужен интернет).

In [ ]:
!pip install -q sentence-transformers

# эмбеддинги предложений; модель скачивается один раз (≈470 МБ)
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("intfloat/multilingual-e5-small")
    for name, a, b in pairs:
        emb = model.encode(["query: " + a, "query: " + b], normalize_embeddings=True)
        print(f"{name:<18} cos = {float(emb[0] @ emb[1]):.3f}")
except Exception as e:
    print("Не удалось загрузить модель (нужен интернет и пакет sentence-transformers):", e)

## 4. Применение: исправление опечаток

Простейший корректор: для слова вне словаря берём слова словаря с наименьшим расстоянием Левенштейна. Ниже словарь из 40 слов (в начальной форме), слова с опечатками и правильные слова в косвенных формах. Последние корректор тоже «исправит», потому что в словаре их нет.

In [ ]:
vocab = ["университет", "студент", "студентка", "преподаватель", "лекция", "практикум", "лаборатория", "библиотека",
         "книга", "учебник", "экзамен", "задание", "вопрос", "ответ", "перевод", "словарь", "предложение", "слово",
         "текст", "язык", "лингвистика", "алгоритм", "программа", "компьютер", "данные", "модель", "расстояние",
         "сходство", "вектор", "матрица", "город", "школа", "тетрадь", "ручка", "таблица", "буква", "символ",
         "строка", "правило", "пример"]
typos = ["универститет", "лингвистка", "превод", "сходсво", "алгоритьм", "матрца", "сллово"]
inflected = ["студентами", "библиотеках", "словарями"]   # правильные слова, но не в начальной форме

def suggest(word, vocab, k=3):
    # лучшие k слов словаря по расстоянию Левенштейна
    return sorted((nltk.edit_distance(word, v), v) for v in vocab)[:k]

for w in typos + inflected:
    print(f"{w:<16}", ", ".join(f"{v} ({d})" for d, v in suggest(w, vocab)))

Для формы «студентами» ближайшие слова находятся только на расстоянии 3 и больше: Левенштейн ничего не знает о морфологии. Решение — сравнивать **основы**: сначала стеммер, потом расстояние до основ словаря.

In [ ]:
stem_vocab = {stem(v): v for v in vocab}

def suggest_stem(word, k=3):
    s = stem(word)
    return s, sorted((nltk.edit_distance(s, sv), v) for sv, v in stem_vocab.items())[:k]

for w in inflected:
    s, cands = suggest_stem(w)
    print(f"{w:<16} → {s:<12}", ", ".join(f"{v} ({d})" for d, v in cands))

## 5. Применение: поиск почти-дубликатов

Новостные агрегаторы склеивают заметки об одном событии. Ниже 10 заголовков: три группы пересказов одного события (с перестановкой слов, другой формой глагола, опечаткой) и посторонние заголовки. Ищем пары с косинусом ≥ 0.5 по словам и по символьным n-граммам.

In [ ]:
heads = [
    "В Самарканде открылась новая библиотека",        # 0
    "В Самарканде открыли новую библиотеку",          # 1
    "Новая библиотека открылась в Самарканде",        # 2
    "В Ташкенте прошёл международный марафон",        # 3
    "Международный марафон прошёл в Ташкенте",        # 4
    "В Ташкенте прошол международный марафон",        # 5
    "Студенты изучают алгоритмы обработки текста",    # 6
    "Студенты изучили алгоритм обработки текстов",    # 7
    "Завтра в Бухаре ожидается жаркая погода",        # 8
    "Цены на хлопок выросли на бирже",                # 9
]
groups = [{0, 1, 2}, {3, 4, 5}, {6, 7}]   # пары внутри группы — дубликаты

word_vec = TfidfVectorizer(tokenizer=tokens, token_pattern=None, lowercase=False)
S_word = cosine_similarity(word_vec.fit_transform(heads))
S_char = cosine_similarity(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(heads))

gold = {(i, j) for g in groups for i in g for j in g if i < j}
threshold = 0.5
for name, S_ in [("по словам", S_word), ("по символьным n-граммам", S_char)]:
    found = {(i, j) for i in range(len(heads)) for j in range(i + 1, len(heads)) if S_[i, j] >= threshold}
    tp = len(found & gold)
    print(f"=== {name}: найдено пар {len(found)}, верных {tp} из {len(gold)}")
    for i, j in sorted(gold | found):
        mark = "✓" if (i, j) in found & gold else ("пропуск" if (i, j) in gold else "ложная")
        print(f"   {S_[i, j]:.2f} {mark:<10} {heads[i]}  ~  {heads[j]}")
    print()

Символьные n-граммы находят пересказы, в которых слова стоят в других формах или с опечаткой. На тепловых картах ниже это видно как более светлые блоки по диагонали.

*Совет:* можно загрузить свой `texts.csv` через панель «Файлы» слева и найти дубликаты в нём (`pd.read_csv("texts.csv")`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, S_, title in [(axes[0], S_word, "Косинус TF-IDF по словам"), (axes[1], S_char, "Косинус TF-IDF по n-граммам символов")]:
    im = ax.imshow(S_, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(heads)))
    ax.set_yticks(range(len(heads)))
    ax.set_title(title)
    for i in range(len(heads)):
        for j in range(len(heads)):
            if i != j and S_[i, j] >= 0.3:
                ax.text(j, i, f"{S_[i, j]:.1f}", ha="center", va="center", fontsize=7,
                        color="black" if S_[i, j] > 0.6 else "white")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

## 6. Задания

1. **Взвешенный Левенштейн.** Перепишите `levenshtein_table`, чтобы замена «похожих» букв (е/ё, и/й, а/о) стоила 0.5. Посчитайте расстояния «ёлка/елка» и «малако/молоко» и сравните с обычным расстоянием.
2. **Дамерау–Левенштейн.** Добавьте в динамику четвёртый вариант — перестановку двух соседних букв со стоимостью 1: `D[i-2][j-2] + 1`, если `a[i-1] == b[j-2]` и `a[i-2] == b[j-1]`. Назовите функцию `damerau(a, b)` (возвращает число); самопроверка уже стоит внизу ячейки задания.
3. Для пар «словоизменение» и «другая тема» из раздела 3 посчитайте Жаккара по символьным n-граммам при $n = 1…6$. При каком $n$ разница между парами наибольшая?
4. Ускорьте корректор из раздела 4: сравнивайте слово только со словами словаря, длина которых отличается не больше чем на 2. Сколько вызовов `edit_distance` вы сэкономили на списке `typos`? Изменились ли подсказки?
5. Для поиска дубликатов (раздел 5) переберите порог от 0.1 до 0.9 с шагом 0.05 отдельно для `S_word` и `S_char` и постройте график F1-меры ($F_1 = 2PR/(P+R)$) от порога. В каком диапазоне порогов каждый вариант находит все дубликаты без ложных?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь



# Самопроверка: запустите ячейку после того, как напишете damerau выше
try:
    for a, b in [("спасибо", "спсаибо"), ("кошка", "окошко"), ("ток", "кот"), ("текст", "тексты")]:
        assert damerau(a, b) == nltk.edit_distance(a, b, transpositions=True), (a, b)
    print("Задание 2: всё верно!")
except NameError:
    print("Сначала определите функцию damerau(a, b) в ячейке задания 2.")

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь